# DeepSeek-OCR-2: Visual Causal Flow

**Paper:** [arXiv:2601.20552](https://arxiv.org/abs/2601.20552)

**Requirements:** GPU runtime (T4 or better), ~15GB GPU memory

## 1. Setup Environment

In [ ]:
# Check GPU
!nvidia-smi

import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Install dependencies
!pip install -q transformers==4.46.3 accelerate pillow addict attrdict einops timm easydict
!pip install -q flash-attn --no-build-isolation

print("✓ Dependencies installed. Restart runtime if needed.")

## 2. Download Sample Images

In [ ]:
import os
import urllib.request

os.makedirs("test-images", exist_ok=True)
os.makedirs("output", exist_ok=True)
os.makedirs("results/sidebyside", exist_ok=True)

GITHUB_RAW = "https://raw.githubusercontent.com/maycuatroi1/ocr-comparison/master/test-images"

sample_images = ["crazy-hand-writing.png", "wild.png", "Document.jpg"]

print("Downloading sample images...")
for img in sample_images:
    try:
        urllib.request.urlretrieve(f"{GITHUB_RAW}/{img}", f"test-images/{img}")
        print(f"  ✓ {img}")
    except Exception as e:
        print(f"  ✗ {img}: {e}")

print(f"\nImages: {os.listdir('test-images')}")

## 3. Load Model

In [ ]:
from transformers import AutoModel, AutoTokenizer
import torch
import os

os.environ["CUDA_VISIBLE_DEVICES"] = '0'
MODEL_NAME = "deepseek-ai/DeepSeek-OCR-2"

print(f"Loading {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

model = AutoModel.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    _attn_implementation='flash_attention_2',
    use_safetensors=True
)
model = model.eval().cuda().to(torch.bfloat16)

print("✓ Model loaded!")

## 4. OCR Functions

In [ ]:
import time
import json
import textwrap
import glob
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display
from datetime import datetime

def perform_ocr(image_path: str, output_dir: str = "output") -> tuple:
    """Perform OCR using DeepSeek-OCR-2"""
    start = time.time()
    
    # Clear output dir
    for f in glob.glob(f"{output_dir}/*.md"):
        os.remove(f)
    
    try:
        # Use official prompt format
        prompt = "<image>\n<|grounding|>Convert the document to markdown."
        
        result = model.infer(
            tokenizer,
            prompt=prompt,
            image_file=image_path,
            output_path=output_dir,
            base_size=1024,
            image_size=768,
            crop_mode=True,
            save_results=True
        )
        
        elapsed = time.time() - start
        
        # Check if result saved to file
        md_files = glob.glob(f"{output_dir}/*.md")
        if md_files:
            with open(md_files[0], 'r', encoding='utf-8') as f:
                text = f.read()
        elif result is not None:
            text = str(result) if not isinstance(result, dict) else result.get('text', str(result))
        else:
            text = "No text extracted"
        
        return text, elapsed
    except Exception as e:
        return f"Error: {str(e)}", time.time() - start


def get_font(size=14):
    try:
        return ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSansMono.ttf", size)
    except:
        return ImageFont.load_default()


def create_sidebyside(image_path, ocr_text, processing_time, output_path):
    """Create side-by-side: Original | OCR Results"""
    original = Image.open(image_path).convert("RGB")
    target_height = 800
    ratio = target_height / original.height
    new_width = int(original.width * ratio)
    original_resized = original.resize((new_width, target_height), Image.Resampling.LANCZOS)

    text_panel_width = 600
    text_panel = Image.new('RGB', (text_panel_width, target_height), (30, 30, 30))
    draw = ImageDraw.Draw(text_panel)

    title_font, text_font, small_font = get_font(20), get_font(14), get_font(12)

    draw.text((15, 15), "OCR Results (DeepSeek-OCR-2)", fill="#FF6B6B", font=title_font)
    draw.line([(15, 45), (text_panel_width - 15, 45)], fill="#FF6B6B", width=2)

    y_offset, line_height, line_num = 60, 20, 1
    for line in ocr_text.split('\n'):
        if not line.strip():
            y_offset += line_height // 2
            continue
        for i, wrapped in enumerate(textwrap.wrap(line, width=55) or ['']):
            if y_offset > target_height - 40:
                draw.text((15, y_offset), "... [truncated]", fill="#888", font=text_font)
                break
            if i == 0:
                draw.text((15, y_offset), f"{line_num:2d}.", fill="#888", font=text_font)
                line_num += 1
            draw.text((50, y_offset), wrapped, fill="white", font=text_font)
            y_offset += line_height
        if y_offset > target_height - 40:
            break

    stats_y = target_height - 30
    draw.line([(15, stats_y - 10), (text_panel_width - 15, stats_y - 10)], fill="#444", width=1)
    num_lines = len([l for l in ocr_text.split('\n') if l.strip()])
    draw.text((15, stats_y), f"⏱ {processing_time:.2f}s | Lines: {num_lines} | Chars: {len(ocr_text)} | DeepSeek-OCR-2",
              fill="#FF6B6B", font=small_font)

    combined = Image.new('RGB', (new_width + 3 + text_panel_width, target_height), (60, 60, 60))
    combined.paste(original_resized, (0, 0))
    ImageDraw.Draw(combined).rectangle([new_width, 0, new_width + 3, target_height], fill="#FF6B6B")
    combined.paste(text_panel, (new_width + 3, 0))

    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    combined.save(str(output_path), quality=95)
    return combined

print("✓ Functions defined")

## 5. Run OCR

In [ ]:
IMAGE_DIR = Path("test-images")
OUTPUT_DIR = Path("results/sidebyside")

image_files = list(IMAGE_DIR.glob("*.png")) + list(IMAGE_DIR.glob("*.jpg")) + list(IMAGE_DIR.glob("*.jpeg"))

print(f"Processing {len(image_files)} images...")
print("=" * 60)

all_results = {}

for img_path in sorted(image_files):
    print(f"\n📷 {img_path.name}")
    
    abs_path = str(img_path.absolute())
    ocr_text, proc_time = perform_ocr(abs_path)
    
    print(f"  ⏱ {proc_time:.2f}s")
    print(f"  Preview: {ocr_text[:80].replace(chr(10), ' ')}...")
    
    out_path = OUTPUT_DIR / f"{img_path.stem}_sidebyside.png"
    vis = create_sidebyside(img_path, ocr_text, proc_time, out_path)
    print(f"  ✓ Saved: {out_path.name}")
    
    display(vis.resize((vis.width // 2, vis.height // 2)))
    
    with open(OUTPUT_DIR / f"{img_path.stem}_ocr.txt", "w") as f:
        f.write(ocr_text)
    
    all_results[img_path.name] = {"text": ocr_text, "time": proc_time, "ts": datetime.now().isoformat()}

with open(OUTPUT_DIR / "deepseek_results.json", "w") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)

print("\n" + "=" * 60)
print("✓ Done!")

## 6. Download Results

In [ ]:
from google.colab import files
!zip -r deepseek_results.zip results/
files.download("deepseek_results.zip")

## 7. Upload Your Own Images

In [ ]:
from google.colab import files

print("Upload images:")
uploaded = files.upload()

for fname in uploaded:
    dest = f"test-images/{fname}"
    if os.path.exists(fname):
        os.rename(fname, dest)
    
    print(f"\n📷 {fname}")
    ocr_text, proc_time = perform_ocr(os.path.abspath(dest))
    print(f"  ⏱ {proc_time:.2f}s")
    
    out_path = OUTPUT_DIR / f"{Path(fname).stem}_sidebyside.png"
    vis = create_sidebyside(dest, ocr_text, proc_time, out_path)
    display(vis.resize((vis.width // 2, vis.height // 2)))
    
    print(f"\nExtracted text:\n{'='*40}\n{ocr_text}")